# Heatmaps: significant genes, transcript isoforms and ORFs

One objective: see whether samples of the same group cluster together, using the rows edgeR found significantly different between the groups (FDR below `alpha`) in the normalized expression matrices the pipeline wrote in S4. Each heatmap is drawn with all samples, and again without the samples named in `exclude`.

In [ ]:
source("src/heatmap.R")
source("src/paths.R")
loc <- run_locations()        # every folder of the run, from paths.json
comparison <- loc$comparison
alpha <- 0.05                  # FDR cutoff for the rows to show; 1 shows every tested row
exclude <- c("420_20190402")   # samples to leave out in the second drawing; character(0) skips it
groups <- read_groups(file.path(loc$leafcutter, paste0(comparison, ".lr_leafcutter.groups_file.txt")))
exclude <- intersect(exclude, groups$sample)   # a sample that is not in this run is ignored
groups

**Explanation.** The line that starts with `loc` reads `paths.json`, the one place where the run's results folder, the comparison name and the dataset name are set. The other cells use short names such as `loc$leafcutter`, so to look at another run you change `paths.json` and nothing in the notebook. The groups file lists each sample and its group. It tells the heatmap which columns to use and how to colour the group bar. `alpha` is the cutoff on the false discovery rate (FDR) from edgeR: only genes, isoforms and ORFs with FDR below it are drawn; set `alpha <- 1` to draw every tested row. `exclude` names the samples to leave out in the second drawing of each heatmap; make it `character(0)` to skip that drawing.

In [ ]:
mat <- read_expression(file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR_normalized_CPM_matrix.txt")),
                       groups, id_col = "isoform_id", label_col = "transcript_name")
orf <- read_expression(file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR_normalized_CPM_matrix.txt")),
                       groups, id_col = "orf_all_isoform_id", label_col = "orf_base_id")
rbind(isoforms = detected_per_sample(mat), orfs = detected_per_sample(orf))

**Explanation.** For each sample, the number of transcript isoforms (first row) and ORFs (second row) with any expression, from the lowest sample to the highest. A sample far below the others has little coverage, and that alone can make its column look different on a heatmap.

In [ ]:
dge_file <- file.path(loc$de_gene, paste0(comparison, "_DGE_edgeR"))
tx_file  <- file.path(loc$de_transcript, paste0(comparison, "_DTE_edgeR"))
orf_file <- file.path(loc$de_orf, paste0(comparison, "_DE_ORF_edgeR"))
genes_sig <- read_expression(paste0(dge_file, "_normalized_CPM_matrix.txt"), groups, id_col = "reference_gene_id", label_col = "gene_name",
  keep = significant_ids(paste0(dge_file, "_results.txt"), "reference_gene_id", alpha))
mat_sig <- read_expression(paste0(tx_file, "_normalized_CPM_matrix.txt"), groups, id_col = "isoform_id", label_col = "transcript_name",
  keep = significant_ids(paste0(tx_file, "_results.txt"), "isoform_id", alpha))
orf_sig <- read_expression(paste0(orf_file, "_normalized_CPM_matrix.txt"), groups, id_col = "orf_all_isoform_id", label_col = "orf_base_id",
  keep = significant_ids(paste0(orf_file, "_results.txt"), "orf_all_isoform_id", alpha))
c(genes = nrow(genes_sig), isoforms = nrow(mat_sig), orfs = nrow(orf_sig))

**Explanation.** For genes, transcript isoforms and ORFs, the rows with FDR below `alpha` in edgeR's results table are kept from the expression matrix. The messages say how many rows of each table passed, and the last line how many rows will be drawn. If a comparison has none, its heatmap says so and is skipped; try a larger `alpha` (for example 0.1) to see the closest rows.

In [ ]:
show_heatmap(expression_heatmap(genes_sig, groups, title = paste0("Genes, FDR < ", alpha, ", all samples")), "gene_heatmap")

**Explanation.** Each row is a gene, each column a sample. Values are log2(CPM + 1) with every row centred and scaled, so colour shows whether a sample is above (red) or below (blue) that gene's average, not how large the gene is. Only genes with FDR below `alpha` are drawn. Rows and columns are grouped by similarity (hierarchical clustering). CPM is counts per million reads.

In [ ]:
show_heatmap(expression_heatmap(mat_sig, groups, title = paste0("Transcript isoforms, FDR < ", alpha, ", all samples")), "isoform_heatmap")

**Explanation.** The same for transcript isoforms: each row is an isoform with FDR below `alpha`.

In [ ]:
show_heatmap(expression_heatmap(orf_sig, groups, title = paste0("ORFs, FDR < ", alpha, ", all samples")), "orf_heatmap")

**Explanation.** The same for open reading frames (ORFs), the predicted protein-coding stretches. Each row is an ORF with FDR below `alpha`, named by its first transcript. Every row has its name, and the figure is as tall as the rows need.

In [ ]:
if (length(exclude) > 0) {
  g2 <- drop_samples(groups, exclude)
  label <- paste(exclude, collapse = ", ")
  show_heatmap(expression_heatmap(genes_sig[, g2$sample, drop = FALSE], g2, title = paste0("Genes, FDR < ", alpha, ", without ", label)), "gene_heatmap_without")
  show_heatmap(expression_heatmap(mat_sig[, g2$sample, drop = FALSE], g2, title = paste0("Transcript isoforms, FDR < ", alpha, ", without ", label)), "isoform_heatmap_without")
  show_heatmap(expression_heatmap(orf_sig[, g2$sample, drop = FALSE], g2, title = paste0("ORFs, FDR < ", alpha, ", without ", label)), "orf_heatmap_without")
} else {
  message("exclude is empty: no second drawing")
}

**Explanation.** The same three heatmaps without the samples named in `exclude`. The rows are the ones significant with all samples, since edgeR was run once with all of them. Compare each with its all-samples drawing: if the groups separate more clearly, or the odd column stops driving the clustering, the excluded sample was influencing the picture. Leaving a sample out after seeing that it looks different needs a reason, so look at the detected counts first and report both versions.